В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [1]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 58.2 MB/s eta 0:00:00


## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [2]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

**Базовые операции с векторами**

In [3]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [4]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


*Ваш ответ здесь*

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [6]:
# 1. Загружаем легкий датасет твитов (чтобы не ждать долго)
twitter_model = api.load('glove-twitter-25')

KeyboardInterrupt: 

2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [7]:
# 2. Функция поиска 10 близких слов
def get_top_10_similar(word, model):
    try:
        similar = model.most_similar(word, topn=10)
        print(f"Слова, похожие на '{word}':")
        for w, score in similar:
            print(f"  {w}: {score:.4f}")
    except KeyError:
        print(f"Слово '{word}' не найдено в словаре.")

get_top_10_similar('computer', twitter_model)

Слова, похожие на 'computer':
  camera: 0.9078
  cell: 0.8919
  server: 0.8745
  device: 0.8694
  wifi: 0.8631
  screen: 0.8622
  app: 0.8616
  case: 0.8588
  remote: 0.8584
  file: 0.8575


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [8]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [10]:
# 3. Обучаем модель Word2Vec на кулинарном датасете
model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,       # размер вектора: 50
    window=3,             # размер окна: 3
    min_count=1,          # минимальная частота слова: 1
    workers=2,            # потоков: 2
    sg=1                  # 1 = использовать skip-gram
)

print("Модель успешно обучена!")
print(f"Слова в словаре: {list(model.wv.key_to_index.keys())[:10]}...")

Модель успешно обучена!
Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


In [11]:
print(f"Слова в словаре: {list(model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [12]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938
  брокколи: 0.1846
  кипятить: 0.1711


In [14]:
# Найдите слова, похожие на "духовка"
try:
    print("\nСлова, похожие на 'духовка':")
    for word, score in model.wv.most_similar('духовка', topn=3):
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово не найдено")

# Найдите слова, похожие на "овощи"
try:
    print("\nСлова, похожие на 'овощи':")
    for word, score in model.wv.most_similar('овощи', topn=3):
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово не найдено")


Слова, похожие на 'духовка':
  ингредиенты: 0.3199
  десерт: 0.3064
  холодильник: 0.2705

Слова, похожие на 'овощи':
  мариновать: 0.2716
  хлеб: 0.2691
  гриль: 0.2546


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [16]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [17]:
print("\n--- FastText ---")
for target_word in ["варить", "духовка", "овощи"]:
    try:
        similar = ft_model.wv.most_similar(target_word, topn=3)
        print(f"Слова, похожие на '{target_word}': {[w for w, _ in similar]}")
    except KeyError:
        print(f"Слово '{target_word}' не найдено")


--- FastText ---
Слова, похожие на 'варить': ['жарить', 'парить', 'месить']
Слова, похожие на 'духовка': ['взбивать', 'лимон', 'салат']
Слова, похожие на 'овощи': ['жарить', 'фольга', 'морковь']


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [19]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('learning')
compare_models('neural')

# Сравниваем модели на словах с опечатками
# Оригиналы: морковь, молоко, духовка
misspelled_words = ['марковь', 'малако', 'духофка']

for word in misspelled_words:
    compare_models(word)


Сравнение для слова: 'learning'
  Word2Vec: слово не найдено
  FastText: ['духовка', 'пирог']

Сравнение для слова: 'neural'
  Word2Vec: слово не найдено
  FastText: ['мука', 'травы']

Сравнение для слова: 'марковь'
  Word2Vec: слово не найдено
  FastText: ['морковь', 'овощи']

Сравнение для слова: 'малако'
  Word2Vec: слово не найдено
  FastText: ['готовить', 'чай']

Сравнение для слова: 'духофка'
  Word2Vec: слово не найдено
  FastText: ['бекон', 'травы']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [20]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [21]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [22]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [23]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [24]:
print("\nСравнение doc_2 и doc_4:")
compare_documents(2, 4)  # python programming vs computer vision


Сравнение doc_2 и doc_4:
Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [25]:
print("\nСамый похожий документ на doc_1:")
most_sim_to_doc1 = doc_model.dv.most_similar("doc_1", topn=2)
# Берем первый элемент (если топ-1 не является самим doc_1)
for doc_tag, score in most_sim_to_doc1:
    if doc_tag != "doc_1":
        doc_id = int(doc_tag.split('_')[1])
        print(f"  {doc_tag} (score: {score:.4f}) -> {documents[doc_id]}")
        break


Самый похожий документ на doc_1:
  doc_0 (score: 0.2735) -> machine learning is interesting


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [26]:
dimensions = [10, 50, 100]
target_words = ['мясо', 'варить', 'тесто']

for size in dimensions:
    print(f"\n--- Модель Word2Vec с vector_size={size} ---")
    test_model = Word2Vec(sentences=cooking_sentences, vector_size=size, window=3, min_count=1, workers=2, sg=1)

    for w in target_words:
        try:
            sims = test_model.wv.most_similar(w, topn=3)
            print(f"[{w}]: {[sim_w for sim_w, _ in sims]}")
        except KeyError:
            pass


--- Модель Word2Vec с vector_size=10 ---
[мясо]: ['жарить', 'курица', 'мука']
[варить]: ['рыба', 'сковорода', 'хлеб']
[тесто]: ['специи', 'тушить', 'барбекю']

--- Модель Word2Vec с vector_size=50 ---
[мясо]: ['завтрак', 'взбивать', 'жарить']
[варить]: ['вино', 'ингредиенты', 'хлеб']
[тесто]: ['жарить', 'масло', 'помидоры']

--- Модель Word2Vec с vector_size=100 ---
[мясо]: ['тост', 'кипятить', 'помидоры']
[варить]: ['чашка', 'вино', 'сковорода']
[тесто]: ['десерт', 'месить', 'горшок']


Результаты эксперимента хорошо иллюстрируют важную особенность Word2Vec:
алгоритм крайне чувствителен к объему данных. Наш обучающий корпус состоит всего из 15 предложений, что для нейросетевых моделей такого типа критически мало.

Переобучение на больших векторах: При размерностях 50 и 100 модель начинает выдавать случайные ассоциации (например, связывает «мясо» и «тост»). В этом случае размерность векторного пространства превышает размер самого словаря. Векторы распределяются слишком разреженно, и их сходство отражает скорее случайный шум, чем реальную семантику.

Плюс малых размерностей: Парадоксально, но наименьший размер вектора (vector_size=10) показал наиболее логичные результаты. Жестко ограниченное пространство заставляет модель активнее сжимать информацию, группируя слова по базовым признакам.